# 과제 1. Dictionary Baseline — 난독화 복원

**목표:** 난독화 복원 문제를 가장 단순한 방식(단어 사전 치환)으로 해결해보며, 문제의 구조를 이해한다.

---
**목차**
1. 라이브러리 설치 & 데이터 로드
2. 난독화 패턴 분석
3. Train / Validation 분리
4. 단어 사전(Dictionary) 구축
5. 문자열 치환 복원 구현
6. Validation 평가 (Accuracy + 문자 단위 F1 Score)
7. Test 데이터 예측 & submission 저장
8. Rule-based 방식 분석 및 한계

---
## 1. 라이브러리 설치 & 데이터 로드

In [ ]:
!pip install pandas scikit-learn -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
import re
from collections import Counter

# 경로 설정
DATA_DIR = "/content/drive/MyDrive/dataset"

TRAIN_CSV      = os.path.join(DATA_DIR, "train.csv")
TEST_CSV       = os.path.join(DATA_DIR, "test.csv")
SUBMISSION_CSV = os.path.join(DATA_DIR, "sample_submission.csv")

# 데이터 로드
train = pd.read_csv(TRAIN_CSV, encoding='utf-8-sig')
test  = pd.read_csv(TEST_CSV,  encoding='utf-8-sig')

print(f"Train 샘플 수: {len(train):,}개 | Test 샘플 수: {len(test):,}개")
print()
print(train.head(3).to_string())

Train 샘플 수: 11,263개 | Test 샘플 수: 1,689개

            ID                                                                                                             input                                                                                                            output
0  TRAIN_00000  별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈닐 탯끎룐눈 녀뮤 퀼교... 야뭍툰 둠 변 닺씨 깍낄 싫훈 굣. 깸삥읊 20여 년 댜녁뵨 곧 중 쩨윌 귑푼 낙팠떤 곶.  별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자니 댓글로는 너무 길고... 아무튼 두 번 다시 가길 싫은 곳. 캠핑을 20여 년 다녀본 곳 중 제일 기분 나빴던 곳.
1  TRAIN_00001                                                                                            잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ                                                                                            잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ
2  TRAIN_00002                                                                                                   절테 간면 않 된는 굣 멥몫                                                                                                   절대 

---
## 2. 난독화 패턴 분석

모델을 만들기 전에 **입력(난독화)과 출력(정상)이 어떻게 대응**되는지 살펴본다.  
패턴을 이해해야 적절한 복원 전략을 선택할 수 있다.

In [ ]:
# ── 2-1. 샘플 5개로 입출력 대응 관계 육안 확인 ──────────────────────────────
print("=" * 70)
print("[입출력 샘플 비교]")
print("=" * 70)
for i in range(5):
    inp = train['input'].iloc[i]
    out = train['output'].iloc[i]
    print(f"\n[{i}] INPUT : {inp[:80]}")
    print(f"    OUTPUT: {out[:80]}")

[입출력 샘플 비교]

[0] INPUT : 별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈닐 탯끎룐눈 녀뮤 퀼교... 야뭍툰 둠 변 닺씨 깍낄 싫훈 굣
    OUTPUT: 별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자니 댓글로는 너무 길고... 아무튼 두 번 다시 가길 싫은 곳

[1] INPUT : 잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ
    OUTPUT: 잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ

[2] INPUT : 절테 간면 않 된는 굣 멥몫
    OUTPUT: 절대 가면 안 되는 곳 메모

[3] INPUT : 야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵겠댜! 한눈 쌀람한뗌많 쭈쳔. 탐패 냄쌕갊 묘둔 쟝졈울 까저갼
    OUTPUT: 아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵겠다! 하는 사람한테만 추천. 담배 냄새가 모든 장점을 가져가

[4] INPUT : 집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 않추 탸툿했숲뉘닫. 휜닉쑵퍅끄왐 걸뤼툐 멂쥐 안았셔 좋앝쿄욥.
    OUTPUT: 지인 추천으로 다녀왔는데 가성비 좋고 깔끔하네요. 저렴한데 물과 음료도 있고 방도 아주 따뜻했습니다. 휘닉스파크와 거리도 멀지 않아서 좋았고요.


In [ ]:
# ── 2-2. 단어 단위 1:1 대응 여부 확인 ──────────────────────────────────────
# 단어 수가 같은 비율 → 단어 정렬 가능 여부 파악

same_len_count = 0
total = len(train)

len_diffs = []
for inp, out in zip(train['input'], train['output']):
    iw = inp.split()
    ow = out.split()
    diff = len(iw) - len(ow)
    len_diffs.append(diff)
    if diff == 0:
        same_len_count += 1

diff_counter = Counter(len_diffs)
same_ratio = same_len_count / total * 100

print(f"입출력 단어 수가 동일한 샘플 비율: {same_ratio:.1f}% ({same_len_count:,}/{total:,})")
print()
print("단어 수 차이 분포 (상위 10개):")
for diff, cnt in sorted(diff_counter.items(), key=lambda x: -x[1])[:10]:
    print(f"  차이 {diff:+d}  →  {cnt:,}건 ({cnt/total*100:.1f}%)")

입출력 단어 수가 동일한 샘플 비율: 100.0% (11,263/11,263)

단어 수 차이 분포 (상위 10개):
  차이 +0  →  11,263건 (100.0%)


In [ ]:
# ── 2-3. 단어 단위 변환 일관성 확인 ────────────────────────────────────────
# '같은 input 단어'가 항상 '같은 output 단어'로 바뀌는지 확인

# 단어 수가 같은 샘플로만 분석
pair_map = {}   # input_word → {output_word: count}

for inp, out in zip(train['input'], train['output']):
    iwords = inp.split()
    owords = out.split()
    if len(iwords) != len(owords):
        continue
    for iw, ow in zip(iwords, owords):
        if iw not in pair_map:
            pair_map[iw] = Counter()
        pair_map[iw][ow] += 1

# 하나의 input 단어가 여러 output 단어로 매핑되는 경우
ambiguous = {k: v for k, v in pair_map.items() if len(v) > 1}
consistent = {k: v for k, v in pair_map.items() if len(v) == 1}

print(f"사전 내 전체 input 단어 수: {len(pair_map):,}개")
print(f"  - 일관된 매핑 (1:1):   {len(consistent):,}개 ({len(consistent)/len(pair_map)*100:.1f}%)")
print(f"  - 모호한 매핑 (1:N):   {len(ambiguous):,}개 ({len(ambiguous)/len(pair_map)*100:.1f}%)")
print()

# 모호한 예시 5개 출력
print("[모호한 매핑 예시 — 같은 난독화 단어가 여러 정상 단어로 대응]")
for k, v in list(ambiguous.items())[:5]:
    print(f"  '{k}'  →  {dict(v)}")

사전 내 전체 input 단어 수: 173,020개
  - 일관된 매핑 (1:1):   171,749개 (99.3%)
  - 모호한 매핑 (1:N):   1,271개 (0.7%)

[모호한 매핑 예시 — 같은 난독화 단어가 여러 정상 단어로 대응]
  '한'  →  {'한': 588, '하': 1}
  '왜'  →  {'왜': 21, '외': 6}
  '펼'  →  {'별': 30, '펄': 1}
  '곧'  →  {'곳': 40, '곧': 4}
  '쟉꼬'  →  {'자고': 1, '작고': 3}


In [ ]:
# ── 2-4. 문자 단위 변형 특성 분석 ───────────────────────────────────────────
# 한 글자씩 비교해 어떤 종류의 변형이 일어났는지 파악

char_same  = 0   # 글자가 그대로인 경우
char_diff  = 0   # 글자가 바뀐 경우
pair_count = 0

for inp, out in zip(train['input'], train['output']):
    iwords = inp.split()
    owords = out.split()
    if len(iwords) != len(owords):
        continue
    for iw, ow in zip(iwords, owords):
        if len(iw) == len(ow):  # 길이가 같은 단어 쌍만
            for ic, oc in zip(iw, ow):
                if ic == oc:
                    char_same += 1
                else:
                    char_diff += 1
                pair_count += 1

print("[글자 단위 변형 통계 (단어 길이가 같은 쌍)]")
print(f"  총 비교 글자 수: {pair_count:,}")
print(f"  변형 없음 (원문 유지): {char_same:,}  ({char_same/pair_count*100:.1f}%)")
print(f"  변형 있음 (다른 글자): {char_diff:,}  ({char_diff/pair_count*100:.1f}%)")
print()
print("[시사점]")
print("  → 난독화는 글자 수준에서 한글 음절 치환 방식으로 이루어진 것으로 보임")
print("  → 단어 길이 자체는 대체로 보존됨 (어절 단위 1:1 대응 가능)")

[글자 단위 변형 통계 (단어 길이가 같은 쌍)]
  총 비교 글자 수: 795,735
  변형 없음 (원문 유지): 162,943  (20.5%)
  변형 있음 (다른 글자): 632,792  (79.5%)

[시사점]
  → 난독화는 글자 수준에서 한글 음절 치환 방식으로 이루어진 것으로 보임
  → 단어 길이 자체는 대체로 보존됨 (어절 단위 1:1 대응 가능)


###  패턴 분석 요약

| 관찰 | 내용 |
|------|------|
| 어절(띄어쓰기 기준) 개수 | 입출력이 대체로 **동일** → 1:1 단어 정렬이 가능 |
| 글자 단위 변형 | 동일 위치 음절이 **다른 한글 음절**로 치환됨 |
| 단어 길이 | 대부분 **보존됨** (글자 수 동일) |
| 매핑 일관성 | 동일 난독화 단어가 여러 정상 단어에 대응되는 **모호한 경우 존재** |

---
## 3. Train / Validation 분리

In [ ]:
from sklearn.model_selection import train_test_split

train_data, val_data = train_test_split(
    train,
    test_size=0.2,
    random_state=42
)

train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f"Train: {len(train_data):,}개 | Validation: {len(val_data):,}개")

Train: 9,010개 | Validation: 2,253개


---
## 4. 단어 사전(Dictionary) 구축

> **아이디어:** 학습 데이터에서 `(난독화 단어, 정상 단어)` 쌍을 추출해 사전을 만든다.  
> 동일한 난독화 단어가 여러 정상 단어에 대응될 경우, **가장 자주 등장한 정상 단어**를 선택한다.

In [ ]:
# ── 4-1. 후보 수집 (빈도 기반) ───────────────────────────────────────────────
pair_freq = {}   # {input_word: Counter({output_word: count})}

for inp, out in zip(train_data['input'], train_data['output']):
    iwords = inp.split()
    owords = out.split()
    # 단어 수가 같은 경우에만 정렬 가능
    if len(iwords) != len(owords):
        continue
    for iw, ow in zip(iwords, owords):
        if iw not in pair_freq:
            pair_freq[iw] = Counter()
        pair_freq[iw][ow] += 1

print(f"수집된 (난독화 → 정상) 후보 단어 쌍: {len(pair_freq):,}개")

수집된 (난독화 → 정상) 후보 단어 쌍: 141,850개


In [ ]:
# ── 4-2. 최빈 단어로 사전 확정 ───────────────────────────────────────────────
match_dict = {iw: counter.most_common(1)[0][0] for iw, counter in pair_freq.items()}

# 통계 출력
ambiguous_keys = [k for k, v in pair_freq.items() if len(v) > 1]
print(f"최종 사전 크기       : {len(match_dict):,}개 항목")
print(f"모호성 해소 필요 항목: {len(ambiguous_keys):,}개 (최빈값 선택으로 처리)")
print()
print("[사전 샘플 10개]")
for k, v in list(match_dict.items())[:10]:
    print(f"  '{k}'  →  '{v}'")

최종 사전 크기       : 141,850개 항목
모호성 해소 필요 항목: 1,013개 (최빈값 선택으로 처리)

[사전 샘플 10개]
  '엷끼카'  →  '여기가'
  '억쿠인울룟'  →  '워크인으로'
  '윕쉴'  →  '입실'
  '같눙한'  →  '가능한'
  '곳윈카욤?'  →  '곳인가요?'
  '없뿔'  →  '어플'
  '옌악운'  →  '예약은'
  '않'  →  '안'
  '됨욤~~'  →  '돼요~~'
  '췻쑈퇴뉘'  →  '취소되니'


---
## 5. 문자열 치환 복원 구현

In [ ]:
def replace_words(input_text: str, match_dict: dict) -> str:
    """
    입력 문장을 단어(어절) 단위로 분리하고,
    사전에 있는 단어는 대응되는 정상 단어로 치환한다.
    사전에 없는 단어는 원문 그대로 유지한다.
    """
    words = input_text.split()
    replaced = [match_dict.get(w, w) for w in words]
    return " ".join(replaced)

In [ ]:
# 복원 예시 확인
print("[복원 예시 3개]")
for i in range(3):
    inp  = train_data['input'].iloc[i]
    gold = train_data['output'].iloc[i]
    pred = replace_words(inp, match_dict)
    print(f"\nINPUT : {inp[:70]}")
    print(f"PRED  : {pred[:70]}")
    print(f"GOLD  : {gold[:70]}")

[복원 예시 3개]

INPUT : 엷끼카 억쿠인울룟 윕쉴 같눙한 곳윈카욤? 없뿔 옌악운 않 됨욤~~ 췻쑈퇴뉘 우위샤향울 공쥐예 헤놓을셔아 할 뜻.
PRED  : 여기가 워크인으로 입실 가능한 곳인가요? 어플 예약은 안 돼요~~ 취소되니 유의사항을 공지에 해놓으셔야 할 듯.
GOLD  : 여기가 워크인으로 입실 가능한 곳인가요? 어플 예약은 안 돼요~~ 취소되니 유의사항을 공지에 해놓으셔야 할 듯.

INPUT : 쌩귄 치 얼먀 앉 뙨 홋퉤립랗 꺌쿰헹욜 윙칟돛 좋교 묫든 쥑건푼듦위 친철핥계 뎃헤 쥬셨써 슉밗한는 똥얀 깊뷴 좋궷 옇헹할 쑤 
PRED  : 생긴 지 얼마 안 된 호텔이라 깔끔해요 위치도 좋고 모든 직원분들이 친절하게 대해 주셔서 숙박하는 동안 기분 좋게 여행할 수 
GOLD  : 생긴 지 얼마 안 된 호텔이라 깔끔해요 위치도 좋고 모든 직원분들이 친절하게 대해 주셔서 숙박하는 동안 기분 좋게 여행할 수 

INPUT : 쭝윈창읫 섶핏수 맛윈두갓 좋쑵닐따. 찐쩔핫쉭교 좋씁뉘타.
PRED  : 주인장의 서비스 마인드가 좋습니다. 친절하시고 좋습니다.
GOLD  : 주인장의 서비스 마인드가 좋습니다. 친절하시고 좋습니다.


---
## 6. Validation 평가

두 가지 지표로 성능을 측정한다.
- **Exact Match Accuracy:** 문장 전체가 완벽하게 일치하는 비율
- **문자 단위 F1 Score:** 예측과 정답 사이의 문자 겹침을 측정 (부분 정답에도 점수 부여)

In [ ]:
# ── 6-1. Validation 예측 ─────────────────────────────────────────────────────
val_pred = val_data['input'].apply(lambda x: replace_words(x, match_dict))

In [ ]:
# ── 6-2. Exact Match Accuracy ────────────────────────────────────────────────
exact_match_accuracy = (val_pred == val_data['output']).mean()
print(f"Exact Match Accuracy: {exact_match_accuracy:.6f}  ({exact_match_accuracy*100:.2f}%)")

Exact Match Accuracy: 0.004882  (0.49%)


In [ ]:
# ── 6-3. 문자 단위 F1 Score ──────────────────────────────────────────────────
# 문자를 하나하나 토큰으로 보고 Precision / Recall / F1을 계산한다.
# (SQuAD 스타일 character-level F1)

def char_f1(pred: str, gold: str) -> float:
    """두 문자열 사이의 문자 단위 F1 Score를 반환한다."""
    pred_chars = Counter(pred.replace(" ", ""))  # 공백 제거 후 문자 빈도
    gold_chars = Counter(gold.replace(" ", ""))

    # 공통 문자 수 (교집합)
    common = sum((pred_chars & gold_chars).values())

    if common == 0:
        return 0.0

    precision = common / sum(pred_chars.values())
    recall    = common / sum(gold_chars.values())
    f1        = 2 * precision * recall / (precision + recall)
    return f1


f1_scores = [
    char_f1(pred, gold)
    for pred, gold in zip(val_pred, val_data['output'])
]

mean_f1 = sum(f1_scores) / len(f1_scores)

print(f"문자 단위 평균 F1 Score: {mean_f1:.6f}  ({mean_f1*100:.2f}%)")
print()
print("[F1 분포]")
buckets = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
for lo, hi in zip(buckets, buckets[1:]):
    cnt = sum(1 for f in f1_scores if lo <= f < hi)
    bar = '█' * (cnt // max(len(f1_scores)//50, 1))
    print(f"  [{lo:.1f}, {hi:.1f})  {cnt:5,}건  {bar}")
# F1 = 1.0인 경우
cnt = sum(1 for f in f1_scores if f == 1.0)
bar = '█' * (cnt // max(len(f1_scores)//50, 1))
print(f"  [1.0, 1.0]  {cnt:5,}건  {bar}")

문자 단위 평균 F1 Score: 0.442914  (44.29%)

[F1 분포]
  [0.0, 0.2)     42건  
  [0.2, 0.4)    797건  █████████████████
  [0.4, 0.6)  1,192건  ██████████████████████████
  [0.6, 0.8)    190건  ████
  [0.8, 1.0)     20건  
  [1.0, 1.0]     12건  


In [ ]:
# ── 6-4. 지표 요약 ───────────────────────────────────────────────────────────
print("=" * 45)
print("        Validation 성능 요약")
print("=" * 45)
print(f"  Exact Match Accuracy : {exact_match_accuracy*100:.2f}%")
print(f"  문자 단위 F1 Score   : {mean_f1*100:.2f}%")
print("=" * 45)
print()
print("[해석]")
print("  - Exact Match가 낮아도 F1이 상대적으로 높다면,")
print("    사전 치환이 '부분적으로는' 올바른 단어를 복원하고 있음을 의미한다.")
print("  - 문장 전체를 완벽히 맞추기 어렵기 때문에 Exact Match는 매우 낮게 나온다.")

        Validation 성능 요약
  Exact Match Accuracy : 0.49%
  문자 단위 F1 Score   : 44.29%

[해석]
  - Exact Match가 낮아도 F1이 상대적으로 높다면,
    사전 치환이 '부분적으로는' 올바른 단어를 복원하고 있음을 의미한다.
  - 문장 전체를 완벽히 맞추기 어렵기 때문에 Exact Match는 매우 낮게 나온다.


---
## 7. Test 데이터 예측 & Submission 생성

In [ ]:
# ── 전체 train으로 사전 재구축 (val도 포함시켜 사전 커버리지 최대화) ──────────
full_pair_freq = {}

for inp, out in zip(train['input'], train['output']):
    iwords = inp.split()
    owords = out.split()
    if len(iwords) != len(owords):
        continue
    for iw, ow in zip(iwords, owords):
        if iw not in full_pair_freq:
            full_pair_freq[iw] = Counter()
        full_pair_freq[iw][ow] += 1

full_match_dict = {iw: cnt.most_common(1)[0][0] for iw, cnt in full_pair_freq.items()}
print(f"전체 train 기반 사전 크기: {len(full_match_dict):,}개")

전체 train 기반 사전 크기: 173,020개


In [ ]:
# test 예측
converted_reviews = test['input'].apply(
    lambda x: replace_words(x, full_match_dict)
).tolist()

print(f"Test 예측 완료: {len(converted_reviews):,}개")
print()
print("[Test 예측 샘플 3개]")
for i in range(3):
    print(f"  INPUT : {test['input'].iloc[i][:60]}")
    print(f"  PRED  : {converted_reviews[i][:60]}")
    print()

Test 예측 완료: 1,689개

[Test 예측 샘플 3개]
  INPUT : 녀뮨넒뭅 만죡숭러윤 효템뤼에오. 푸싸눼 옰면 콕 츄쩐학꼬 싶은 콧쉰웨오. 췌꾜윕뉘댜! ㅎㅎ 당음웨 또 옭 컷
  PRED  : 녀뮨넒뭅 만죡숭러윤 효템뤼에오. 푸싸눼 오면 꼭 츄쩐학꼬 싶은 콧쉰웨오. 췌꾜윕뉘댜! ㅎㅎ 다음에 또 올 것

  INPUT : 풀룐투갸 엎코, 좀식또 업읍머, 윌뱐 잎츔민든릿 샤있샤윔엡 위썬 호뗄첨렴 관뤽갉 찰 앉 뙨는 누뀜뮈넬오. 까
  PRED  : 풀룐투갸 없고, 좀식또 업읍머, 일반 잎츔민든릿 샤있샤윔엡 있어 호뗄첨렴 관리가 잘 안 되는 누뀜뮈넬오. 까

  INPUT : 쥔차 붉찐졀행욘. 삶먼섶 멂묽럿턴 혹텔 중웨 쬐약위였습뉜따. 칙어뉜쥐 샤쨩윈쥐 쩨끄윈할 땝붇텄 찐쩔함 1됴 
  PRED  : 진짜 붉찐졀행욘. 삶먼섶 멂묽럿턴 호텔 중에 쬐약위였습뉜따. 칙어뉜쥐 샤쨩윈쥐 쩨끄윈할 땝붇텄 찐쩔함 1도 



In [ ]:
# Submission 저장
submission = pd.read_csv(SUBMISSION_CSV, encoding='utf-8-sig')
submission['output'] = converted_reviews
submission.to_csv('./submission_dictionary.csv', index=False, encoding='utf-8-sig')

print("submission_dictionary.csv 저장 완료!")
print(submission.head(3).to_string())

submission_dictionary.csv 저장 완료!
          ID                                                                                                                                                                                                                                                                                                                                                                    output
0  TEST_0000                                                                                                                                                                                                                                                                                                         녀뮨넒뭅 만죡숭러윤 효템뤼에오. 푸싸눼 오면 꼭 츄쩐학꼬 싶은 콧쉰웨오. 췌꾜윕뉘댜! ㅎㅎ 다음에 또 올 것 갗았요.
1  TEST_0001                                                                                                                                                                                                             

---
## 8. Rule-based 방식 분석

###  Rule-based(사전 치환) 방식의 장점

| 장점 | 설명 |
|------|------|
| **구현 단순성** | 복잡한 모델 없이 Python 딕셔너리 조회만으로 동작한다. |
| **속도** | GPU 불필요. 수만 건 데이터를 밀리초 단위로 처리한다. |
| **해석 가능성** | 어떤 단어가 왜 바뀌었는지 사전을 직접 확인할 수 있다. |
| **결정론적 예측** | 같은 입력에 항상 같은 출력을 보장한다. |
| **학습 데이터 의존성 낮음** | 소량 데이터만 있어도 사전을 구성할 수 있다. |

---

###  Rule-based 방식의 한계

| 한계 | 설명 |
|------|------|
| **OOV 문제** | 학습 데이터에 없는 난독화 단어(Out-Of-Vocabulary)는 복원 불가 → 원문 그대로 남음 |
| **문맥 무시** | 같은 난독화 단어도 문맥에 따라 다른 정상 단어일 수 있으나, 사전은 하나의 고정 값만 반환함 |
| **부분 매핑 불가** | 단어 수가 다른 입출력 쌍(삽입·삭제 포함)은 사전 구축 자체가 불가능 |
| **오염된 사전** | 학습 데이터의 노이즈(잘못된 쌍)가 그대로 사전에 반영됨 |

---

###  왜 일반화 성능이 낮은가?

```
핵심 원인 1 — OOV (Out-Of-Vocabulary)
  사전은 train에서 본 단어쌍만 저장한다.
  val/test에 처음 등장하는 난독화 단어는 복원되지 않고 난독화 상태로 남는다.
  → Exact Match 입장에서는 단 하나의 단어만 틀려도 0점

핵심 원인 2 — 문맥 독립성
  '닮패' → '담배' vs '닮패' → '냄새' 처럼
  같은 난독화 단어가 문맥에 따라 다른 정상 단어에 대응될 수 있다.
  사전 기반 방법은 문맥을 전혀 보지 않으므로 항상 최빈 단어만 선택한다.

핵심 원인 3 — 어절 수 불일치
  일부 샘플은 입출력 어절 수가 달라 1:1 정렬 자체가 불가능하다.
  이런 샘플들은 사전 구축 단계에서 제외되어 사전 커버리지가 낮아진다.
```

---

### 어떤 난독화 패턴에 강한가?

```
강한 패턴
  ① 단어 단위 1:1 고정 치환
     동일한 난독화 단어가 항상 동일한 정상 단어로 대응되는 경우
     예: '잚많' → '잠만', '쟉꼬' → '자고' (고정 매핑)

  ② 단어 개수 보존
     입출력 어절 수가 같아 위치 기반 정렬이 가능한 경우
     사전에 등록된 단어라면 위치와 무관하게 정확히 복원됨

  ③ 고빈도 단어
     train에 많이 등장한 난독화 단어일수록 사전 정확도가 높음
     (최빈값 선택의 신뢰도 상승)

약한 패턴
  ① 동음이의어/다의어 난독화
     문맥에 따라 다른 단어로 복원되어야 하는 경우
  ② 어절 수 변동
     조사 분리, 복합어 분해 등으로 어절 수가 달라지는 경우
  ③ 미등록 단어 (OOV)
     train에서 한 번도 보지 못한 난독화 단어
```

In [ ]:
# ── 정리: 최종 성능 요약 출력 ────────────────────────────────────────────────
print("=" * 55)
print("     Dictionary Baseline — 최종 결과 요약")
print("=" * 55)
print(f"  사전 크기 (train 전체 기반) : {len(full_match_dict):,}개")
print(f"  Validation Exact Match     : {exact_match_accuracy*100:.4f}%")
print(f"  Validation 문자 F1 Score   : {mean_f1*100:.4f}%")
print("=" * 55)
print()
print("[결론]")
print("  Rule-based 방식은 구현이 간단하고 빠르지만,")
print("  OOV와 문맥 무시 문제로 인해 Exact Match 기준 성능은 매우 낮다.")
print("  이 baseline을 출발점으로, 언어 모델(LLM) 기반 접근으로")
print("  성능을 크게 향상시킬 수 있다.")

     Dictionary Baseline — 최종 결과 요약
  사전 크기 (train 전체 기반) : 173,020개
  Validation Exact Match     : 0.4882%
  Validation 문자 F1 Score   : 44.2914%

[결론]
  Rule-based 방식은 구현이 간단하고 빠르지만,
  OOV와 문맥 무시 문제로 인해 Exact Match 기준 성능은 매우 낮다.
  이 baseline을 출발점으로, 언어 모델(LLM) 기반 접근으로
  성능을 크게 향상시킬 수 있다.


---
## 9. 오류 분석 (과제 3 — Error Analysis)

Dictionary Baseline이 **어디서, 왜, 어떻게 실패하는지** 정량·정성적으로 분석한다.  
이 분석 결과는 LLM Baseline 개선 방향을 결정하는 데 직접 활용된다.

| 분석 항목 | 측정 지표 / 방법 |
|-----------|------------------|
| 추가 지표 | BLEU Score, Edit Distance (CER) |
| OOV 분석 | val에서 사전에 없는 단어 비율 |
| 실패 사례 | F1이 낮은 샘플 top-10 |
| 패턴 분석 | 자주 틀리는 난독화 단어 top-20 |

In [ ]:
!pip install nltk -q
import nltk
nltk.download('punkt', quiet=True)
print("설치 완료")

설치 완료


### 9-1. BLEU Score

BLEU(Bilingual Evaluation Understudy)는 기계 번역 평가에서 출발한 지표로,  
예측 문장과 정답 문장 사이의 **n-gram 겹침 비율**을 측정한다.

- **n-gram**: 연속된 n개의 문자(여기서는 문자 단위 적용)
- 1에 가까울수록 정답에 가깝고, 0에 가까울수록 완전히 틀림
- 문자 F1과 달리 **순서 정보**까지 반영함

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def compute_char_bleu(pred: str, gold: str) -> float:
    """
    문자 단위 BLEU Score를 계산한다.
    - 공백 제거 후 한 글자씩 토큰으로 취급
    - smoothing: 짧은 문장에서 0이 되는 것을 방지
    """
    pred_chars = list(pred.replace(" ", ""))
    gold_chars = list(gold.replace(" ", ""))
    if len(pred_chars) == 0 or len(gold_chars) == 0:
        return 0.0
    smoother = SmoothingFunction().method1
    return sentence_bleu(
        [gold_chars],          # 정답 (리스트의 리스트)
        pred_chars,            # 예측
        weights=(0.5, 0.5),    # bigram까지만 (한글 특성상 bigram이 적합)
        smoothing_function=smoother
    )

bleu_scores = [
    compute_char_bleu(pred, gold)
    for pred, gold in zip(val_pred, val_data['output'])
]

mean_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"문자 단위 평균 BLEU Score: {mean_bleu:.6f}  ({mean_bleu*100:.2f}%)")
print()
print("[BLEU 분포]")
buckets = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
for lo, hi in zip(buckets, buckets[1:]):
    cnt = sum(1 for b in bleu_scores if lo <= b < hi)
    bar = '█' * (cnt // max(len(bleu_scores)//50, 1))
    print(f"  [{lo:.1f}, {hi:.1f})  {cnt:5,}건  {bar}")
cnt = sum(1 for b in bleu_scores if b == 1.0)
bar = '█' * (cnt // max(len(bleu_scores)//50, 1))
print(f"  [1.0, 1.0]  {cnt:5,}건  {bar}")

문자 단위 평균 BLEU Score: 0.344890  (34.49%)

[BLEU 분포]
  [0.0, 0.2)    257건  █████
  [0.2, 0.4)  1,357건  ██████████████████████████████
  [0.4, 0.6)    525건  ███████████
  [0.6, 0.8)     94건  ██
  [0.8, 1.0)      8건  
  [1.0, 1.0]     12건  


### 9-2. Edit Distance & CER (Character Error Rate)

**Edit Distance(편집 거리)**는 한 문자열을 다른 문자열로 바꾸는 데 필요한 최소 연산 수(삽입·삭제·교체)이다.

**CER(Character Error Rate)** = Edit Distance / 정답 문자 수
- 0에 가까울수록 정확한 복원
- 1 이상이면 정답 길이보다 오류가 많음
- F1·BLEU와 달리 **얼마나 많이 고쳐야 하는지** 직관적으로 보여줌

In [ ]:
def edit_distance(s1: str, s2: str) -> int:
    """동적 프로그래밍으로 두 문자열 사이의 편집 거리를 계산한다."""
    s1 = s1.replace(" ", "")
    s2 = s2.replace(" ", "")
    m, n = len(s1), len(s2)
    # dp[i][j] = s1[:i]를 s2[:j]로 바꾸는 최소 연산 수
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j],    # 삭제
                                    dp[i][j-1],    # 삽입
                                    dp[i-1][j-1])  # 교체
    return dp[m][n]


def cer(pred: str, gold: str) -> float:
    """Character Error Rate = Edit Distance / 정답 문자 수"""
    gold_len = len(gold.replace(" ", ""))
    if gold_len == 0:
        return 0.0
    return edit_distance(pred, gold) / gold_len


# 샘플이 많으면 느릴 수 있으므로 최대 500개만 계산 (시간 절약)
sample_size = min(500, len(val_data))
val_sample_pred = val_pred.iloc[:sample_size].tolist()
val_sample_gold = val_data['output'].iloc[:sample_size].tolist()

cer_scores = [cer(p, g) for p, g in zip(val_sample_pred, val_sample_gold)]
edit_dists = [edit_distance(p, g) for p, g in zip(val_sample_pred, val_sample_gold)]

mean_cer  = sum(cer_scores) / len(cer_scores)
mean_edit = sum(edit_dists) / len(edit_dists)

print(f"평균 Edit Distance : {mean_edit:.2f} 글자")
print(f"평균 CER           : {mean_cer:.4f}  ({mean_cer*100:.2f}%)")
print()
print("[CER 해석]")
print(f"  → 예측 문장 1개당 평균 {mean_edit:.1f}글자를 고쳐야 정답이 됨")
if mean_cer < 0.2:
    print("  → CER < 20%: 부분적으로 꽤 잘 복원되고 있음")
elif mean_cer < 0.5:
    print("  → CER 20~50%: 절반 정도는 맞추지만 개선 여지 큼")
else:
    print("  → CER > 50%: 복원 품질이 낮음 — LLM 도입 필요")

평균 Edit Distance : 39.71 글자
평균 CER           : 0.5677  (56.77%)

[CER 해석]
  → 예측 문장 1개당 평균 39.7글자를 고쳐야 정답이 됨
  → CER > 50%: 복원 품질이 낮음 — LLM 도입 필요


### 9-3. OOV(Out-Of-Vocabulary) 분석

사전에 없는 단어가 얼마나 되는지 측정한다.  
OOV 단어는 **사전 치환이 불가능**하므로 난독화 상태 그대로 출력된다.  
→ OOV 비율이 높을수록 Rule-based 방식의 한계가 두드러진다.

In [ ]:
# val 데이터에서 OOV 단어 비율 측정
total_words  = 0
oov_words    = 0
oov_examples = []  # (단어, 해당 문장) 샘플 수집

for inp in val_data['input']:
    for word in inp.split():
        total_words += 1
        if word not in match_dict:
            oov_words += 1
            if len(oov_examples) < 30:
                oov_examples.append(word)

oov_ratio = oov_words / total_words * 100

print(f"[Validation OOV 분석]")
print(f"  총 단어 수      : {total_words:,}개")
print(f"  OOV 단어 수     : {oov_words:,}개")
print(f"  OOV 비율        : {oov_ratio:.2f}%")
print()
print("[OOV 단어 예시 (앞 20개)]")
print("  ", oov_examples[:20])
print()
print("[해석]")
print(f"  → 단어 {oov_ratio:.1f}%는 사전에 없어 난독화 상태 그대로 출력됨")
print("  → OOV 단어 하나만 있어도 Exact Match는 0점 → 전반적 성능 저하의 주원인")

[Validation OOV 분석]
  총 단어 수      : 52,416개
  OOV 단어 수     : 31,644개
  OOV 비율        : 60.37%

[OOV 단어 예시 (앞 20개)]
   ['온션퓨', '쑥쏘웩오!', '별섶', '펀쳅', '빵물닒예옷!', '힐륑하꾜', '깝뉘댜ㅎㅎ', '옜푸꼲', '쥑건뿐둘또', '췬졀햐쎄오', '밧댜예셔', '갖캅귄', '태뮨엔', '빳타갔', '뽀이쥐', '굣윕님타.', '칵깝임', '잊여됴', '쑬묩깐', '엽승뉜']

[해석]
  → 단어 60.4%는 사전에 없어 난독화 상태 그대로 출력됨
  → OOV 단어 하나만 있어도 Exact Match는 0점 → 전반적 성능 저하의 주원인


### 9-4. 실패 사례 분석 (F1이 낮은 샘플 Top-10)

F1 Score가 가장 낮은 샘플을 직접 들여다보며  
**어떤 상황에서 복원이 가장 크게 실패하는지** 패턴을 파악한다.

In [ ]:
# F1 Score 기준 정렬하여 최악 10개 출력
val_result_df = val_data.copy().reset_index(drop=True)
val_result_df['pred']      = val_pred.values
val_result_df['f1']        = f1_scores
val_result_df['bleu']      = bleu_scores

worst_10 = val_result_df.nsmallest(10, 'f1')

print("=" * 70)
print("          F1이 가장 낮은 샘플 Top-10")
print("=" * 70)
for rank, (_, row) in enumerate(worst_10.iterrows(), 1):
    print(f"\n[{rank}위] F1={row['f1']:.4f}  BLEU={row['bleu']:.4f}")
    print(f"  INPUT  : {row['input'][:80]}")
    print(f"  PRED   : {row['pred'][:80]}")
    print(f"  GOLD   : {row['output'][:80]}")

          F1이 가장 낮은 샘플 Top-10

[1위] F1=0.0588  BLEU=0.0192
  INPUT  : 게창 쪽귄눈 엿숟선헤쓺냐 침굼문 쪼항오
  PRED   : 게창 쪽귄눈 엿숟선헤쓺냐 침굼문 쪼항오
  GOLD   : 개장 초기는 어수선했으나 지금은 좋아요

[2위] F1=0.0625  BLEU=0.0204
  INPUT  : 윅꾸쳄뀨뛰쁠륨 섶빗수 쿰칙한넷욥.
  PRED   : 윅꾸쳄뀨뛰쁠륨 섶빗수 쿰칙한넷욥.
  GOLD   : 이그제큐티브룸 서비스 끔찍하네요.

[3위] F1=0.0625  BLEU=0.0204
  INPUT  : 샴쟝뉨 췬졀옜 뎁졉팥눈 낍뷰닢닐타.
  PRED   : 샴쟝뉨 췬졀옜 뎁졉팥눈 낍뷰닢닐타.
  GOLD   : 사장님 친절에 대접받는 기분입니다.

[4위] F1=0.0769  BLEU=0.0253
  INPUT  : 캑끝햐콤 셥퓟쓰갉 콕끕윌탔.
  PRED   : 캑끝햐콤 셥퓟쓰갉 콕끕윌탔.
  GOLD   : 깨끗하고 서비스가 고급이다.

[5위] F1=0.0909  BLEU=0.0302
  INPUT  : 눔슥웽셔 봇콕 놂랏눼옮.
  PRED   : 눔슥웽셔 봇콕 놂랏눼옮.
  GOLD   : 뉴스에서 보고 놀랐네요.

[6위] F1=0.0909  BLEU=0.0302
  INPUT  : 짬 작키웬 납프쮜 얀학욜.
  PRED   : 참 작키웬 납프쮜 얀학욜.
  GOLD   : 잠 자기엔 나쁘지 않아요.

[7위] F1=0.1000  BLEU=0.0186
  INPUT  : 셩쑤엌 쿄압읽랐 윈취룰 깜얀한먼 걋썽핑갸 꿴챤앝섶욤. 쪼옹학쿄용.
  PRED   : 셩쑤엌 쿄압읽랐 윈취룰 깜얀한먼 걋썽핑갸 꿴챤앝섶욤. 쪼옹학쿄용.
  GOLD   : 성수역 코앞이라 위치를 감안하면 가성비가 괜찮았어요. 조용하고요.

[8위] F1=0.1111  BLEU=0.0256
  INPUT  : 쉬썲있 께꿋햐곯 치컨뜰뤼 췬절할씹뉘타.
  PRED   : 쉬썲있 께꿋햐곯 치컨뜰뤼 췬절할씹뉘타.
  GOLD   : 시

### 9-5. 자주 틀리는 난독화 단어 패턴 Top-20

어떤 난독화 단어가 가장 많이 틀리는지 집계한다.  
- **틀린 경우**: 예측 단어 ≠ 정답 단어
- 고빈도 오류 단어를 파악하면 → 추가 규칙이나 예외 처리로 성능 개선 가능

In [ ]:
# 단어 단위 오류 집계
# (난독화 단어, 예측 단어, 정답 단어) 튜플을 수집
error_counter  = Counter()  # 오류 발생 횟수
error_examples = {}         # 오류 사례 저장

for inp, pred, gold in zip(
    val_data['input'], val_pred, val_data['output']
):
    inp_words  = inp.split()
    pred_words = pred.split()
    gold_words = gold.split()

    # 단어 수가 같아야 정렬 비교 가능
    if len(inp_words) != len(pred_words) or len(inp_words) != len(gold_words):
        continue

    for iw, pw, gw in zip(inp_words, pred_words, gold_words):
        if pw != gw:  # 예측이 틀린 경우
            error_counter[(iw, pw, gw)] += 1
            if (iw, pw, gw) not in error_examples:
                error_examples[(iw, pw, gw)] = True

# 오류 빈도 기준 정렬
top_errors = error_counter.most_common(20)

print("=" * 65)
print("  자주 틀리는 난독화 단어 Top-20")
print("=" * 65)
print(f"{'순위':>4}  {'난독화(input)':^12}  {'예측(pred)':^12}  {'정답(gold)':^12}  {'횟수':>5}")
print("-" * 65)
for rank, ((iw, pw, gw), cnt) in enumerate(top_errors, 1):
    print(f"  {rank:>2}.  {iw:^12}  {pw:^12}  {gw:^12}  {cnt:>5}회")

  자주 틀리는 난독화 단어 Top-20
  순위   난독화(input)     예측(pred)      정답(gold)       횟수
-----------------------------------------------------------------
   1.       똔             돈             또            8회
   2.       딱             딱             탁            5회
   3.       톤             돈             또            4회
   4.       쑬             쓸             수            4회
   5.       잃             이             일            3회
   6.      툴륌.           드림.           들림.           3회
   7.       둘             두             둘            3회
   8.      햐는찌           햐는찌           하는지           3회
   9.       땋             탕             다            3회
  10.      위썽셔           위썽셔           있어서           3회
  11.       방관            방관            방과           3회
  12.       빻             방             빵            3회
  13.      움...          움...          음...          3회
  14.       판             판             반            3회
  15.       훔             후             흠            3회
  16.       펑읾   

In [ ]:
# 오류를 세 가지 유형으로 분류
# 1) OOV 오류: 사전에 없어서 난독화 단어 그대로 출력된 경우 (iw == pw)
# 2) 모호성 오류: 사전에 있지만 틀린 단어로 매핑된 경우 (iw != pw, iw in dict)
# 3) 어절불일치: 단어 수가 달라 비교 불가

type_oov   = 0  # 사전 미등재
type_ambig = 0  # 사전 등재 but 틀린 매핑
type_match = 0  # 정답
type_skip  = 0  # 어절 수 불일치로 비교 불가

for inp, pred, gold in zip(
    val_data['input'], val_pred, val_data['output']
):
    inp_w  = inp.split()
    pred_w = pred.split()
    gold_w = gold.split()

    if len(inp_w) != len(gold_w):
        type_skip += len(inp_w)
        continue

    for iw, pw, gw in zip(inp_w, pred_w, gold_w):
        if pw == gw:
            type_match += 1
        elif iw == pw:  # 사전에 없어 그대로 출력됨
            type_oov += 1
        else:           # 사전에 있지만 다른 단어로 복원됨
            type_ambig += 1

total_classified = type_oov + type_ambig + type_match + type_skip

print("=" * 50)
print("         단어 단위 오류 유형 분류")
print("=" * 50)
print(f"  ✅ 정확히 복원됨        : {type_match:>6,}개  ({type_match/total_classified*100:.1f}%)")
print(f"  ❌ OOV (사전 미등재)    : {type_oov:>6,}개  ({type_oov/total_classified*100:.1f}%)")
print(f"  ⚠️  모호성 오류 (잘못매핑): {type_ambig:>6,}개  ({type_ambig/total_classified*100:.1f}%)")
print(f"  ⏭️  어절수 불일치 (skip)  : {type_skip:>6,}개  ({type_skip/total_classified*100:.1f}%)")
print("=" * 50)
print()
print("[오류 원인 비중 해석]")
dominant = max(
    ('OOV', type_oov),
    ('모호성', type_ambig),
    ('어절불일치', type_skip),
    key=lambda x: x[1]
)
print(f"  → 가장 큰 오류 원인: '{dominant[0]}' ({dominant[1]:,}개)")
print(f"  → OOV는 사전 확장으로 줄일 수 있으나, 모호성은 문맥 인식 모델(LLM)이 필요")

         단어 단위 오류 유형 분류
  ✅ 정확히 복원됨        : 20,471개  (39.1%)
  ❌ OOV (사전 미등재)    : 31,402개  (59.9%)
  ⚠️  모호성 오류 (잘못매핑):    543개  (1.0%)
  ⏭️  어절수 불일치 (skip)  :      0개  (0.0%)

[오류 원인 비중 해석]
  → 가장 큰 오류 원인: 'OOV' (31,402개)
  → OOV는 사전 확장으로 줄일 수 있으나, 모호성은 문맥 인식 모델(LLM)이 필요


### 📊 오류 분석 종합 요약

| 분석 항목 | 결과 | 시사점 |
|-----------|------|--------|
| **BLEU Score** | (실행 후 기록) | n-gram 순서까지 고려한 복원 정확도 |
| **CER** | (실행 후 기록) | 글자 단위로 얼마나 고쳐야 하는지 |
| **OOV 비율** | (실행 후 기록) | 사전 부재로 인한 복원 실패 비율 |
| **최다 오류 단어** | (실행 후 기록) | 집중적으로 개선할 대상 |
| **오류 유형 1위** | OOV or 모호성 | LLM이 필요한 근거 |

---

### 🔗 오류 분석 → LLM 개선 방향 연결

```
오류 분석에서 발견된 문제점          →   LLM에서의 해결 방식
─────────────────────────────────────────────────────────────
OOV 단어 복원 불가                  →   pretrained 언어 모델은 대규모
                                        어휘로 미등록 단어도 처리 가능

문맥 무시로 인한 모호성 오류         →   encoder가 전체 문장 문맥을 인코딩,
                                        decoder가 문맥에 맞는 단어 생성

어절 수 불일치 샘플 처리 불가        →   seq2seq 구조로 길이 제약 없이
                                        자유롭게 생성 가능

자주 틀리는 고빈도 오류 단어         →   fine-tuning 데이터에 해당 패턴을
                                        충분히 포함시켜 학습
```